# Multiclass Fish Image Classification
Run all cells to automatically extract the data, train the models, find the best one, and download `best_model.h5`.

In [ ]:
import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive
from google.colab import files

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.applications import VGG16, ResNet50, MobileNet, InceptionV3, EfficientNetB0

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

# Update this path if Dataset.zip is in a different folder in your Drive
zip_path = '/content/drive/MyDrive/Dataset.zip'
extract_path = '/content/Dataset'

if not os.path.exists(extract_path):
    print("Extracting dataset...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Extraction complete.")
else:
    print("Dataset already extracted.")

In [ ]:
# Automatically find the actual dataset folder containing the class subfolders
def get_dataset_dir(base_path):
    for root, dirs, files in os.walk(base_path):
        # If there are multiple directories and no zip files, it's likely the class folder level
        if len(dirs) > 2:
            return root
    return base_path

dataset_dir = get_dataset_dir(extract_path)
print(f"Using dataset directory: {dataset_dir}")

In [ ]:
batch_size = 32
img_height = 224
img_width = 224

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

print("Loading Training Data...")
train_generator = train_datagen.flow_from_directory(
    dataset_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

print("Loading Validation Data...")
validation_generator = train_datagen.flow_from_directory(
    dataset_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

num_classes = train_generator.num_classes

In [ ]:
# Dictionary to store model accuracies to compare later
model_performance = {}
trained_models = {}

def plot_history(history, title):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(history.history['accuracy'], label='Train')
    ax1.plot(history.history['val_accuracy'], label='Validation')
    ax1.set_title(title + ' - Accuracy')
    ax1.legend()
    ax2.plot(history.history['loss'], label='Train')
    ax2.plot(history.history['val_loss'], label='Validation')
    ax2.set_title(title + ' - Loss')
    ax2.legend()
    plt.show()

In [ ]:
# 1. Custom CNN
print("--- Training Custom CNN ---")
cnn_model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(img_height, img_width, 3)),
    MaxPooling2D(2, 2),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])
cnn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train Custom CNN (using fewer epochs to save time, you can increase this)
epochs = 5
history_cnn = cnn_model.fit(train_generator, validation_data=validation_generator, epochs=epochs)

val_acc = max(history_cnn.history['val_accuracy'])
model_performance['Custom_CNN'] = val_acc
trained_models['Custom_CNN'] = cnn_model
plot_history(history_cnn, "Custom CNN")

In [ ]:
# 2. Transfer Learning Models
def create_transfer_model(base_model):
    base_model.trainable = False
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)
    predictions = Dense(num_classes, activation='softmax')(x)
    model = Model(inputs=base_model.input, outputs=predictions)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

base_models = {
    'VGG16': VGG16(weights='imagenet', include_top=False, input_shape=(img_height, img_width, 3)),
    'ResNet50': ResNet50(weights='imagenet', include_top=False, input_shape=(img_height, img_width, 3)),
    'MobileNet': MobileNet(weights='imagenet', include_top=False, input_shape=(img_height, img_width, 3)),
    'InceptionV3': InceptionV3(weights='imagenet', include_top=False, input_shape=(img_height, img_width, 3)),
    'EfficientNetB0': EfficientNetB0(weights='imagenet', include_top=False, input_shape=(img_height, img_width, 3))
}

for name, base_model in base_models.items():
    print(f"\n--- Training {name} ---")
    model = create_transfer_model(base_model)
    history = model.fit(train_generator, validation_data=validation_generator, epochs=epochs)
    
    val_acc = max(history.history['val_accuracy'])
    model_performance[name] = val_acc
    trained_models[name] = model
    plot_history(history, name)

In [ ]:
# 3. Compare and Save Best Model
print("--- Model Performance Comparison ---")
for name, acc in model_performance.items():
    print(f"{name}: Validation Accuracy = {acc:.4f}")

best_model_name = max(model_performance, key=model_performance.get)
print(f"\nBest Model is {best_model_name} with validation accuracy of {model_performance[best_model_name]:.4f}")

# Save the best model
best_model = trained_models[best_model_name]
best_model.save('best_model.h5')
print("Saved best_model.h5 successfully.")

# Automatically download the model
files.download('best_model.h5')